# Predicting Student Test Scores 
##  Score: 8.73686

In [1]:
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error


In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.015,
    'n_estimators': 9000,
    'num_leaves': 89,
    'max_depth': 11,
    'min_child_samples': 46,
    'reg_alpha': 9.4,
    'reg_lambda': 0.34,
    'min_split_gain': 1e-6,
    'subsample': 0.70,
    'subsample_freq': 3,
    'colsample_bytree': 0.62,
    'n_jobs': -1,
    'force_col_wise': True
}

model_variants = [
    ('gbdt', {}),
    ('gbdt_extra_trees', {'extra_trees': True, 'feature_fraction_bynode': 0.75})
]

seeds = [420, 666, 80085]
n_splits = 10

y_bins = pd.qcut(y, q=20, labels=False, duplicates='drop')

all_oof = np.zeros(len(X), dtype=float)
all_test = np.zeros(len(X_test), dtype=float)

n_total_models = len(model_variants) * len(seeds)

for v_i, (variant_name, variant_overrides) in enumerate(model_variants, start=1):
    print(f'VARIANT {variant_name} ({v_i}/{len(model_variants)})')

    variant_oof = np.zeros(len(X), dtype=float)
    variant_test = np.zeros(len(X_test), dtype=float)

    for s_i, seed in enumerate(seeds, start=1):
        params = {**base_params, **variant_overrides, 'random_state': seed}

        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        oof = np.zeros(len(X), dtype=float)
        test_pred = np.zeros(len(X_test), dtype=float)
        rmse_scores = []

        print(f'SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y_bins), start=1):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(350), lgb.log_evaluation(0)]
            )

            va_pred = model.predict(X_va)
            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f}')

            test_pred += model.predict(X_test) / n_splits

        oof = np.clip(oof, 0, 100)
        test_pred = np.clip(test_pred, 0, 100)

        oof_rmse = float(np.sqrt(mean_squared_error(y, oof)))
        print(f'{variant_name} seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        variant_oof += oof / len(seeds)
        variant_test += test_pred / len(seeds)

    variant_oof = np.clip(variant_oof, 0, 100)
    variant_test = np.clip(variant_test, 0, 100)

    variant_rmse = float(np.sqrt(mean_squared_error(y, variant_oof)))
    print(f'{variant_name} FINAL (avg seeds) OOF RMSE: {variant_rmse:.5f}')

    all_oof += variant_oof / len(model_variants)
    all_test += variant_test / len(model_variants)

all_oof = np.clip(all_oof, 0, 100)
all_test = np.clip(all_test, 0, 100)

final_oof_rmse = float(np.sqrt(mean_squared_error(y, all_oof)))
print(f'ENSEMBLE FINAL OOF RMSE: {final_oof_rmse:.5f}')

submission = pd.DataFrame({'id': test_ids, 'exam_score': all_test})
submission.to_csv('submission.csv', index=False)
print('Wrote submission.csv')


VARIANT gbdt (1/2)
SEED 420 (1/3)
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[2862]	valid_0's rmse: 8.72811
  Fold 1/10 RMSE: 8.72811
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[3368]	valid_0's rmse: 8.71895
  Fold 2/10 RMSE: 8.71895
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[4254]	valid_0's rmse: 8.75446
  Fold 3/10 RMSE: 8.75446
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[4040]	valid_0's rmse: 8.74308
  Fold 4/10 RMSE: 8.74308
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[3774]	valid_0's rmse: 8.77698
  Fold 5/10 RMSE: 8.77698
Training until validation scores don't improve for 350 rounds
Early stopping, best iteration is:
[3160]	valid_0's rmse: 8.7715
  Fold 6/10 RMSE: 8.77150
Training until validation scores don't impr